In [7]:
# Capstone Project 3
# Smart Marketing Prediction System (ML Pipeline Project)
# Scenario
# A fast-growing e-commerce company called ShopEasy is struggling with inefficient marketing campaigns.
# Every day thousands of users visit their website. The marketing team spends a large amount of money showing ads, discounts, and promotional emails, but they don't know which customers are actually likely to buy something.
# Currently:
# Many customers browse but never purchase
# Marketing money is wasted on the wrong users
# The company wants to predict purchase probability
# The data science team has been asked to build a machine learning system that predicts whether a customer will purchase a product during a session.
# If the system predicts high probability of purchase, the system will:
# show personalized product recommendations
# offer targeted discounts
# prioritize marketing campaigns
# If the system predicts low probability, the company will avoid spending marketing resources.
# However, the dataset contains both numerical and categorical features, so the data science team must design a complete ML pipeline.
# Dataset is available in DatasetCapstoneProject3 in the github repo link https://github.com/himanshusar123/Datasets
# Business Objective
# Build a machine learning model that predicts whether a user will purchase (1) or not purchase (0) during a website session.
# The model must be implemented using scikit-learn pipelines, including:
# Encoding techniques
# Feature preprocessing
# Model training
# Model selection
# Hyperparameter tuning
# Import required libraries


import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# Load the dataset from Excel file
df = pd.read_excel("/content/DatasetCapstoneProject3.xlsx")


# Separate features (X) and target variable (y)
# CustomerID is removed because it is just an identifier
X = df.drop(["CustomerID","Purchased"], axis=1)
y = df["Purchased"]


# Define numerical and categorical feature columns
numerical_features = ["Age","Time_on_Website","Pages_Visited","Ad_Clicks","Previous_Purchases"]
categorical_features = ["Gender","Device","Traffic_Source"]


# Create pipeline for numerical data
numerical_pipeline = Pipeline([

    # Replace missing values with median
    ("imputer", SimpleImputer(strategy="median")),

    # Standardize numerical values (mean = 0, std = 1)
    ("scaler", StandardScaler())
])


# Create pipeline for categorical data
categorical_pipeline = Pipeline([

    # Replace missing categorical values with most frequent value
    ("imputer", SimpleImputer(strategy="most_frequent")),

    # Convert categorical values into numerical form using OneHotEncoding
    # handle_unknown="ignore" avoids errors for new categories
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


# Combine numerical and categorical preprocessing
# ColumnTransformer applies different preprocessing to different columns
preprocessor = ColumnTransformer([

    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])


# Create complete ML pipeline
# First preprocessing happens, then model training
pipeline = Pipeline([

    ("preprocessing", preprocessor),
    ("model", LogisticRegression())
])


# Define hyperparameters for model tuning
# We test both Logistic Regression and Random Forest
param_grid = [

    # Parameters for Logistic Regression
    {
        "model":[LogisticRegression(max_iter=1000)],
        "model__C":[0.01,0.1,1,10]
    },

    # Parameters for Random Forest
    {
        "model":[RandomForestClassifier()],
        "model__n_estimators":[50,100,200],
        "model__max_depth":[3,5,10]
    }
]


# Split dataset into training and testing sets
# 80% training and 20% testing
# stratify=y keeps class balance
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)


# GridSearchCV is used to find the best model and best parameters
# cv=5 means 5-fold cross validation
grid = GridSearchCV(pipeline,param_grid,cv=5,scoring="accuracy")


# Train the models using training data
grid.fit(X_train,y_train)


# Get the best model selected by GridSearch
best_model = grid.best_estimator_


# Make predictions on test data
y_pred = best_model.predict(X_test)


# Print the best model and parameters
print("Best Model:", grid.best_params_)



# Print model accuracy
print("Accuracy:",accuracy_score(y_test,y_pred))


# Print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test,y_pred))


# Print classification report (precision, recall, f1-score)
print("Classification Report:")
print(classification_report(y_test,y_pred))

# Example: Predict purchase for a new user session

new_user = pd.DataFrame({
    "Age":[30],
    "Gender":["Female"],
    "Device":["Mobile"],
    "Traffic_Source":["Social Media"],
    "Time_on_Website":[15],
    "Pages_Visited":[5],
    "Ad_Clicks":[2],
    "Previous_Purchases":[1]
})

prediction = best_model.predict(new_user)

print("Prediction (0 = No Purchase, 1 = Purchase):", prediction[0])
probability = best_model.predict_proba(new_user)

print("Purchase Probability:", probability)



Best Model: {'model': LogisticRegression(max_iter=1000), 'model__C': 1}
Accuracy: 1.0
Confusion Matrix:
[[1 0]
 [0 2]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3

Prediction (0 = No Purchase, 1 = Purchase): 1
Purchase Probability: [[0.45074688 0.54925312]]
